# Vision Emotion Recognition — FER+ soft labels, modern CNN

**Pipeline:**
1. Read fer2013.csv (pixel strings) + fer2013new.csv (10-voter labels)
2. Merge by row index; drop Disgust, Contempt, Unknown, Not-Face
3. Build soft labels over 6 classes: Angry, Fear, Happy, Neutral, Sad, Surprise
4. Train modern CNN (VGG-style blocks + BatchNorm + GAP) with heavy augmentation
5. Evaluate on FER2013 PublicTest set (held out)
6. Save .keras model + metadata

**Target:** test accuracy ≥ 65% (up from ~51% baseline).

Final model is small (~500K params, ~2MB) so it fits into the real-time pipeline.

## Step 0 — Mount Drive, check GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = '/content/drive/MyDrive/BitirmeProjesi/vision_ferplus'
os.makedirs(OUTPUT_DIR, exist_ok=True)

FER2013_CSV    = '/content/drive/MyDrive/BitirmeProjesi/fer2013.csv'
FERPLUS_CSV    = '/content/drive/MyDrive/BitirmeProjesi/fer2013new.csv'

print(f'fer2013.csv exists: {os.path.exists(FER2013_CSV)}')
print(f'fer2013new.csv exists: {os.path.exists(FERPLUS_CSV)}')

import tensorflow as tf
print(f'TF: {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPU: {gpus}')

## Step 1 — Inspect the two CSV files

Before merging, verify their shapes match and preview a few rows so we know what we're dealing with.

In [ ]:
import pandas as pd

fer = pd.read_csv(FER2013_CSV)
ferplus = pd.read_csv(FERPLUS_CSV)

print('=== fer2013.csv ===')
print(f'Shape: {fer.shape}')
print(f'Columns: {list(fer.columns)}')
print(fer.head(2))
print()
print('=== fer2013new.csv ===')
print(f'Shape: {ferplus.shape}')
print(f'Columns: {list(ferplus.columns)}')
print(ferplus.head(2))

## Step 2 — Merge, filter, build soft labels

**Row alignment:** fer2013new.csv is aligned 1:1 with fer2013.csv — same order. We drop rows where fer2013new couldn't provide a valid label (e.g. empty pixel rows).

**FER+ columns (vote counts):** neutral, happiness, surprise, sadness, anger, disgust, fear, contempt, unknown, NF

**Our mapping (6 classes):**
- 0: Angry    ← anger
- 1: Fear     ← fear
- 2: Happy    ← happiness
- 3: Neutral  ← neutral
- 4: Sad      ← sadness
- 5: Surprise ← surprise

**Filtering rules:** drop a sample if (a) majority vote was disgust/contempt/unknown/NF, OR (b) votes across our 6 target classes sum to 0.

**Soft labels:** for each remaining sample, normalize the 6-class vote counts to sum to 1.0 → probability distribution.

In [ ]:
import numpy as np

# FER+ column order (from Microsoft repo)
FERPLUS_EMOTION_COLS = [
    'neutral', 'happiness', 'surprise', 'sadness',
    'anger', 'disgust', 'fear', 'contempt', 'unknown', 'NF'
]

# Our 6 target classes in the order we want for the model output
CLASSES = ['Angry', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']
# Map target index → ferplus column name
TARGET_TO_FERPLUS = {
    0: 'anger',
    1: 'fear',
    2: 'happiness',
    3: 'neutral',
    4: 'sadness',
    5: 'surprise',
}
DROP_COLS = ['disgust', 'contempt', 'unknown', 'NF']

assert len(fer) == len(ferplus), f'Length mismatch: {len(fer)} vs {len(ferplus)}'

# Confirm required columns are present
for col in FERPLUS_EMOTION_COLS:
    assert col in ferplus.columns, f'Missing column: {col}'

# Build a merged frame with what we need
merged = pd.DataFrame({
    'pixels': fer['pixels'].values,
    'usage': fer['Usage'].values,
})
for col in FERPLUS_EMOTION_COLS:
    merged[col] = ferplus[col].values

# Drop rows where pixels are empty / NaN
merged = merged[merged['pixels'].notna()]
merged = merged[merged['pixels'].str.len() > 0]

# Drop if majority vote was in excluded classes
vote_all = merged[FERPLUS_EMOTION_COLS].values
majority_col_idx = np.argmax(vote_all, axis=1)
majority_col_name = np.array(FERPLUS_EMOTION_COLS)[majority_col_idx]
keep_mask = ~np.isin(majority_col_name, DROP_COLS)
merged = merged[keep_mask].reset_index(drop=True)

# Build soft label: take only our 6 target columns, normalize to sum=1
target_cols_in_order = [TARGET_TO_FERPLUS[i] for i in range(len(CLASSES))]
soft = merged[target_cols_in_order].values.astype(np.float32)
row_sums = soft.sum(axis=1, keepdims=True)
# Extra safety: drop rows where sum is 0 (no votes in any of our 6 classes)
valid = (row_sums.squeeze() > 0)
merged = merged[valid].reset_index(drop=True)
soft = soft[valid]
soft = soft / soft.sum(axis=1, keepdims=True)

print(f'Final dataset size: {len(merged)}')
print(f'Soft label shape: {soft.shape}')
print()
print('Hard label distribution (argmax of soft):')
hard = soft.argmax(axis=1)
for i, c in enumerate(CLASSES):
    print(f'  {c:10s}: {(hard == i).sum():5d}')
print()
print('Usage split (FER2013 original):')
print(merged['usage'].value_counts())

## Step 3 — Decode pixels into images

FER2013's `pixels` column stores each image as a space-separated string of 2304 integers (48×48 grayscale). We convert these to `(N, 48, 48, 1)` float32 arrays.

In [ ]:
from tqdm.auto import tqdm

IMG_SIZE = 48

def decode_pixels(pixel_string):
    arr = np.fromstring(pixel_string, sep=' ', dtype=np.uint8)
    return arr.reshape(IMG_SIZE, IMG_SIZE)

print('Decoding all pixel strings...')
X_all = np.zeros((len(merged), IMG_SIZE, IMG_SIZE, 1), dtype=np.uint8)
for i, px in enumerate(tqdm(merged['pixels'].values)):
    X_all[i, :, :, 0] = decode_pixels(px)

y_all = soft  # already (N, 6) soft labels
usage = merged['usage'].values

print(f'X_all shape: {X_all.shape}, dtype: {X_all.dtype}')
print(f'y_all shape: {y_all.shape}, dtype: {y_all.dtype}')

# FER2013 official split:
# - Training: training set
# - PublicTest: validation set (used for leaderboard)
# - PrivateTest: final test set (held out)
train_mask = usage == 'Training'
val_mask   = usage == 'PublicTest'
test_mask  = usage == 'PrivateTest'

X_train, y_train = X_all[train_mask], y_all[train_mask]
X_val,   y_val   = X_all[val_mask],   y_all[val_mask]
X_test,  y_test  = X_all[test_mask],  y_all[test_mask]

print(f'Train: {X_train.shape}')
print(f'Val:   {X_val.shape}')
print(f'Test:  {X_test.shape}')

## Step 4 — Sanity check: visualize a few samples

Always look at the data before training. Confirms we decoded correctly and the labels make sense.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 6, figsize=(15, 5))
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(X_train), 12, replace=False)
for ax, idx in zip(axes.flat, sample_idx):
    ax.imshow(X_train[idx, :, :, 0], cmap='gray')
    hard_label = CLASSES[y_train[idx].argmax()]
    confidence = y_train[idx].max()
    ax.set_title(f'{hard_label} ({confidence:.2f})', fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Step 5 — Build tf.data pipeline with augmentation

Augmentation applied **only** on training set. Each epoch sees different variants of each image.

- RandomFlip horizontal
- RandomRotation ±10°
- RandomZoom ±10%
- RandomTranslation ±8%
- RandomContrast ±20%
- Rescaling (1/255)

In [ ]:
from tensorflow.keras import layers, models

BATCH_SIZE = 64

augment = models.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.10),
    layers.RandomZoom(0.10),
    layers.RandomTranslation(0.08, 0.08),
    layers.RandomContrast(0.20),
], name='augment')

def make_ds(X, y, training):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if training:
        ds = ds.shuffle(buffer_size=len(X), reshuffle_each_iteration=True)
    ds = ds.batch(BATCH_SIZE)
    if training:
        ds = ds.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(lambda x, y: (tf.cast(x, tf.float32) / 255.0, y), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_ds(X_train, y_train, training=True)
val_ds   = make_ds(X_val,   y_val,   training=False)
test_ds  = make_ds(X_test,  y_test,  training=False)

# Smoke test
for xb, yb in train_ds.take(1):
    print(f'Batch X: {xb.shape}, dtype {xb.dtype}, range [{xb.numpy().min():.3f}, {xb.numpy().max():.3f}]')
    print(f'Batch y: {yb.shape}, sum per sample: {yb.numpy().sum(axis=1)[:3]}')

## Step 6 — Define the modern CNN

**Design choices:**
- **Double-conv blocks:** two 3×3 convs per block before pooling. More expressive than single conv, used in VGG, ResNet.
- **BatchNorm after every conv:** stabilizes training, allows higher learning rates.
- **4 stages** with filter doubling: 32 → 64 → 128 → 256.
- **Global Average Pooling** instead of Flatten: fewer parameters, much less overfitting.
- **Small Dense head** (128 units) before softmax.

Total params: ~500K. Disk size when saved: ~2MB.

In [ ]:
def conv_block(x, filters, name):
    x = layers.Conv2D(filters, 3, padding='same', name=f'{name}_conv1')(x)
    x = layers.BatchNormalization(name=f'{name}_bn1')(x)
    x = layers.Activation('relu', name=f'{name}_act1')(x)
    x = layers.Conv2D(filters, 3, padding='same', name=f'{name}_conv2')(x)
    x = layers.BatchNormalization(name=f'{name}_bn2')(x)
    x = layers.Activation('relu', name=f'{name}_act2')(x)
    x = layers.MaxPool2D(2, name=f'{name}_pool')(x)
    return x

def build_model(num_classes=6):
    inp = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1), name='input')

    x = conv_block(inp, 32,  'block1')   # 48 -> 24
    x = layers.Dropout(0.20)(x)
    x = conv_block(x,   64,  'block2')   # 24 -> 12
    x = layers.Dropout(0.25)(x)
    x = conv_block(x,   128, 'block3')   # 12 -> 6
    x = layers.Dropout(0.30)(x)
    x = conv_block(x,   256, 'block4')   # 6  -> 3
    x = layers.Dropout(0.35)(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(num_classes, activation='softmax', name='output')(x)

    return models.Model(inp, out, name='vision_ferplus_cnn')

model = build_model(num_classes=len(CLASSES))
model.summary()
print(f'\nTotal params: {model.count_params():,}')

## Step 7 — Compile and train

**Loss:** `CategoricalCrossentropy` with `label_smoothing=0.05`. We already have soft labels from FER+; label smoothing adds a tiny extra regularizer.

**Callbacks:**
- EarlyStopping on val_accuracy, patience 15, restore best weights
- ReduceLROnPlateau on val_loss, factor 0.5, patience 5
- ModelCheckpoint saves best val_accuracy to Drive continuously

Expected epoch time on Colab T4: ~15-25s. Convergence typically 40-80 epochs.

In [ ]:
from tensorflow.keras import callbacks

CKPT_PATH = os.path.join(OUTPUT_DIR, 'vision_ferplus_best.keras')

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=[tf.keras.metrics.CategoricalAccuracy(name='accuracy')],
)

cb_list = [
    callbacks.EarlyStopping(
        monitor='val_accuracy', patience=15,
        restore_best_weights=True, mode='max'),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=5, min_lr=1e-5),
    callbacks.ModelCheckpoint(
        CKPT_PATH, monitor='val_accuracy',
        save_best_only=True, mode='max'),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=120,
    callbacks=cb_list,
    verbose=2,
)

## Step 8 — Evaluate

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.plot(history.history['accuracy'], label='train')
a1.plot(history.history['val_accuracy'], label='val')
a1.set_title('Accuracy'); a1.legend(); a1.grid(True)
a2.plot(history.history['loss'], label='train')
a2.plot(history.history['val_loss'], label='val')
a2.set_title('Loss'); a2.legend(); a2.grid(True)
plt.show()

val_loss,  val_acc  = model.evaluate(val_ds,  verbose=0)
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f'Val  accuracy: {val_acc:.4f}')
print(f'Test accuracy: {test_acc:.4f}')

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Gather predictions on the held-out test set
y_pred_probs = model.predict(test_ds, verbose=0)
y_pred = y_pred_probs.argmax(axis=1)
y_true = y_test.argmax(axis=1)  # hard labels from soft argmax

report = classification_report(y_true, y_pred, target_names=CLASSES, digits=3)
print(report)

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel('Predicted'); plt.ylabel('True')
plt.title(f'Vision FER+ Test CM (acc={test_acc:.3f})')
plt.show()

## Step 9 — Save artifacts

In [ ]:
import json
import shutil

FINAL_PATH = os.path.join(OUTPUT_DIR, 'vision_ferplus.keras')
if os.path.exists(CKPT_PATH):
    shutil.copy(CKPT_PATH, FINAL_PATH)
else:
    model.save(FINAL_PATH)
print(f'Final model: {FINAL_PATH}')

size_mb = os.path.getsize(FINAL_PATH) / (1024 * 1024)
print(f'Size on disk: {size_mb:.2f} MB')

meta = {
    'classes': CLASSES,
    'input_shape': [IMG_SIZE, IMG_SIZE, 1],
    'preprocessing': {
        'grayscale': True,
        'size': IMG_SIZE,
        'scale': '/ 255.0 (float32)',
    },
    'val_accuracy': float(val_acc),
    'test_accuracy': float(test_acc),
    'num_train': int(len(X_train)),
    'num_val':   int(len(X_val)),
    'num_test':  int(len(X_test)),
    'labels': 'FER+ soft labels (voter distributions)',
    'loss': 'CategoricalCrossentropy(label_smoothing=0.05)',
    'architecture': '4x(DoubleConv+BN+Pool+Dropout) -> GAP -> Dense(128,BN) -> Softmax(6)',
    'params': int(model.count_params()),
}
with open(os.path.join(OUTPUT_DIR, 'vision_ferplus_meta.json'), 'w') as f:
    json.dump(meta, f, indent=2)
with open(os.path.join(OUTPUT_DIR, 'vision_ferplus_report.txt'), 'w') as f:
    f.write(f'Val accuracy: {val_acc:.4f}\n')
    f.write(f'Test accuracy: {test_acc:.4f}\n\n')
    f.write(report)

print('Metadata + report saved.')

## What to check

1. **Test accuracy ≥ 0.65** → success, proceed to pipeline integration
2. **Train/val gap < 10 points** → no serious overfit
3. **Confusion matrix diagonal is dominant** → no class collapse
4. **Per-class recall ≥ 0.5 for all classes** except maybe Surprise if underrepresented

If test accuracy is below 0.60, consider:
- Training longer (increase patience)
- Stronger augmentation
- Fallback: try the ViT transfer-learning route